# Beacon and Network Path Analysis

> **Safety boundary:** This workbook performs static analysis of sanitized excerpts, fake fixtures, and derived public metadata. It does not execute attacker code, inspect the analyst's environment, or contact any network.

## Beacon Model

The captured stage serialized host and process information into query
parameters named `sysInfo`, `processInfo`, `tid`, and `sysId`, then polled a
direct-IP endpoint. Query strings are especially visible in proxy, gateway, and
server logs and may leak data beyond the destination application.

The real stage also described conditional task evaluation when a response
reported an error state. That creates arbitrary follow-on capability, but is
not reproduced here.

In [ ]:
const base = new URL("http://127.0.0.1:9001/api/checkStatus");
const fakeSysInfo = { hostname: "training-host", platform: "linux", mac: "00:00:00:00:00:00" };
const fakeProcessInfo = { NODE_ENV: "development", npm_lifecycle_event: "prepare" };
base.search = new URLSearchParams({
  sysInfo: JSON.stringify(fakeSysInfo),
  processInfo: JSON.stringify(fakeProcessInfo),
  tid: "fake-token",
  sysId: "fake-system-id",
}).toString();
base.toString();

The cell constructs a loopback URL object only; it does not fetch it.

| Beacon field | Meaning | Risk |
|---|---|---|
| `sysInfo` | host and network profile | fingerprinting and victim selection |
| `processInfo` | process environment snapshot | secret and configuration exposure |
| `tid` | task or victim token | tracking and task correlation |
| `sysId` | stable system identifier | repeat-host correlation |

## Observed Network Failure

The reconstructed runtime trace included:

```text
TypeError: fetch failed
cause: connect ENETUNREACH 136.243.22.62:1224
```

Node's `fetch` implementation uses undici. For a direct IP, no DNS lookup is
needed. The stack attempts a TCP connection, consults the route table, and fails
before connection establishment when the destination is blackholed or otherwise
unreachable.

```text
stage-two fetch
  -> undici request machinery
  -> TCP connect(136.243.22.62, 1224)
  -> route lookup
  -> local blackhole / unreachable route
  -> ENETUNREACH
```

In [ ]:
const observation = {
  runtimeError: "TypeError: fetch failed",
  causeCode: "ENETUNREACH",
  syscall: "connect",
  transport: "TCP",
  connectionEstablished: false,
  applicationResponseReceived: false,
};

const interpretation = observation.causeCode === "ENETUNREACH"
  ? "route selection failed before the observed TCP connection completed"
  : "requires separate analysis";

({ observation, interpretation });

| Network observation | Interpretation | Confidence |
|---|---|---|
| direct IP and port embedded in stage | intended command endpoint | high, static |
| `connect ENETUNREACH` | observed TCP connection did not complete | high, runtime |
| no application response in trace | no observed tasking on that attempt | high for captured run |
| absence of full PCAP | other attempts cannot be excluded | bounded uncertainty |

The failure weighs strongly against completed direct-IP C2 transmission during
the observed attempt. It does not retroactively disprove the earlier serverless
POST, which used a different destination and produced a stage-two response.

**Packet capture would prove:** SYN attempts, replies, completed handshakes, and
visible payload bytes.

**Proxy logs would prove:** URL, timing, status, and potentially query data for
proxied HTTP requests.

**Process telemetry would prove:** which PID initiated each socket and its npm
ancestry.

**Control:** egress policy plus a blackhole route blocked the direct-IP path.
**Detection:** alert on Node under npm ancestry connecting directly to rare
Internet IPs and nonstandard ports.